In [1]:
import re
import os
import glob

In [13]:
def extract_section(md_content, start_markers, stop_markers, length=None):
    if not md_content:
        return ""

    # Normalisasi input
    if isinstance(start_markers, str):
        start_markers = [start_markers]
    if isinstance(stop_markers, str):
        stop_markers = [stop_markers]

    # Gabungkan marker
    start_combined = "|".join(re.escape(m) for m in start_markers)
    stop_combined = "|".join(re.escape(m) for m in stop_markers)

    # Regex start:
    # - tidak wajib awal baris
    # - toleran bold/header
    start_pattern = re.compile(
        rf"(?i)(?:^|\n|\|)\s*[_*#\s]*\b(?:{start_combined})\b[_*\s]*(?:\([^)]*\))?",
        re.MULTILINE
    )

    matches = list(start_pattern.finditer(md_content))
    if not matches:
        return ""

    start_match = matches[-1]

    # start_match = start_pattern.search(md_content)
    if not start_match:
        return ""

    # Mulai setelah judul start
    content_start = start_match.end()

    # Regex stop (judul berikutnya)
    stop_pattern = re.compile(
        rf"(?i)(?:^|\n|\|)\s*[_*#\s]*\b(?:{stop_combined})\b[_*\s]*(?:\([^)]*\))?",
        re.MULTILINE
    )

    stop_match = stop_pattern.search(md_content, content_start)

    result = md_content[content_start:stop_match.start()] if stop_match else md_content[content_start:]
    result = result.strip()
    return result[:length] if length else result


In [25]:
def get_section(md_text, start_pattern, max_length=5000):
    """
    Mengekstrak teks mulai dari pola judul yang ditemukan hingga batas karakter tertentu.
    Sangat kebal terhadap dokumen OCR yang formatnya berantakan.
    """
    if not md_text:
        return ""

    # 1. Cari titik AWAL (Judul Bab)
    start_match = re.search(start_pattern, md_text, re.IGNORECASE | re.MULTILINE)
    if not start_match:
        return "" # Mengembalikan string kosong jika bab tidak ditemukan

    # Titik mulai teks adalah tepat setelah baris judul bab tersebut
    content_start = start_match.end()

    # 2. Ambil semua sisa teks dari titik mulai sampai dokumen habis
    section_content = md_text[content_start:]

    # 3. Bersihkan spasi kosong atau enter di awal dan akhir
    section_content = section_content.strip()

    # 4. Terapkan pemotongan batas karakter (max_length)
    if max_length is not None and len(section_content) > max_length:
        section_content = section_content[:max_length]

    section_content = section_content.replace("|", " ")

    section_content = re.sub(r'-{2,}', ' ', section_content)

    section_content = re.sub(r' {2,}', ' ', section_content)

    section_content = "\n".join([line.rstrip() for line in section_content.splitlines()])

    section_content = section_content.strip()

    return section_content

In [ ]:
DATA_FOLDER_PATH = "knowledge-base/processed"
OUTPUT_FOLDER_PATH = "knowledge-base/temp"

os.makedirs(OUTPUT_FOLDER_PATH, exist_ok=True)

# Cari semua file .md
list_file = glob.glob(os.path.join(DATA_FOLDER_PATH, "*.md"))

print(f"Ditemukan {len(list_file)} file markdown. Memulai ekstraksi...\n")

# collection = []
counter = []
for file_path in list_file:
    filename = os.path.basename(file_path)
    filename_txt = os.path.splitext(filename)[0] + ".txt"  # abc.txt
    output_path = os.path.join(OUTPUT_FOLDER_PATH, filename_txt)

    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    hasil = get_section(raw_text, ["Nutrition Care Process",
                                    "Nutritional Care Process",
                                    "NUTRITIONAL CARE PROCESS",
                                    'Form Nutritional Care Process',
                                    'II. Form Nutritional Care Process',
                                    'BAB II. NCP',
                                    'FORM NUTRITIONAL CARE PROSES'
                                    ],
                                    ["BAB 3",
                                    "BAB III",
                                    'Perhitungan Kebutuhan',
                                    'RENCANA INTERVENSI'
                                    ], length=5000)

    print(filename, "===")
    print('\n')
    # print(hasil[:100],'...', "\n\n")
    if len(hasil) == 0:
        counter.append(filename)
        print('=========')
        print(filename)
        print('=========')
        print(hasil, '\n\n')
    else:
        print(len(hasil))

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(hasil)

print(len(counter))

Ditemukan 120 file markdown. Memulai ekstraksi...

(HIL) DEXTRA INKASERATA + MECKEL’S DIVERTICUM POST EXPLORASI .md ===


75
Abses Paru Post Evakuasi Abses + Wedge-repaired-ocr.md ===


75
ANAK DENGAN TB PARU , PNEUMONIA , GIZI BURUK MARASMUS.md ===


75
ANEMIA + SUSP LANGERHANS CELL HISTIOCYTOSIS + SUSP TU OTAK.md ===


75
ARTHRALGIA + ANEMIA + HIPONATREMIA.md ===


ARTHRALGIA + ANEMIA + HIPONATREMIA.md
 


ASMA BRONKIAL.md ===


75
ASTHMA SERANGAN BERAT, PNEUMONIA, INTRACTABLE EPILEPSI.md ===


75
BAYI DENGAN NEONATUS ATERM + BERAT BADAN LAHIR RENDAH .md ===


34746
BAYI PREMATUR BBLR SMK (SESUAI MASA KEHAMILAN) LAHIR SC 3334 MINGGU.md ===


75
BBLR, OEIS COMPLEX (OMPHALOCELE + EXTROFIA CLOACA + IMPERFRATA ANUS.md ===


75
BERAT BADAN LAHIR SANGAT RENDAH (BBLSR), SESUAI MASA KEHAMILAN (SMK).md ===


75
BERAT BADAN LAHIR SANGAT RENDAH (BBSLR), NEONATUS VERY PETERM (NVP).md ===


75
BONE PAIN DT METASTATIC DISEASE DD RADICULOPHATY, DM TIPE 2.md ===


75
BRONKOPNEUMONIA + GIZI KURANG + 

In [30]:
DATA_FOLDER_PATH = "knowledge-base/processed"
OUTPUT_FOLDER_PATH = "knowledge-base/temp"
pola_ncp = r"^(?:##\s*)?(?:BAB\s*[I1l\|]+\.?\s*)?(?:FORM\s*)?(?:NUTRITION(?:AL)?\s+CARE\s+PRO[CS]ESS?|NCP).*$"

os.makedirs(OUTPUT_FOLDER_PATH, exist_ok=True)

# Cari semua file .md
list_file = glob.glob(os.path.join(DATA_FOLDER_PATH, "*.md"))

print(f"Ditemukan {len(list_file)} file markdown. Memulai ekstraksi...\n")

# collection = []
counter = []
for file_path in list_file:
    filename = os.path.basename(file_path)
    filename_txt = os.path.splitext(filename)[0] + ".txt"  # abc.txt
    output_path = os.path.join(OUTPUT_FOLDER_PATH, filename_txt)

    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    hasil = get_section(raw_text, pola_ncp, max_length=6000)

    print(filename, "===")
    print('\n')
    # print(hasil[:100],'...', "\n\n")
    if len(hasil) == 0:
        counter.append(filename)
        print('=========')
        print(filename)
        print('=========')
        print(hasil, '\n\n')
    else:
        print(len(hasil))

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(hasil)

print(len(counter))

Ditemukan 105 file markdown. Memulai ekstraksi...

(HIL) DEXTRA INKASERATA + MECKEL’S DIVERTICUM POST EXPLORASI .md ===


841
Abses Paru Post Evakuasi Abses + Wedge-repaired-ocr.md ===


4594
ARTHRALGIA + ANEMIA + HIPONATREMIA.md ===


3604
ASMA BRONKIAL.md ===


3060
ASTHMA SERANGAN BERAT, PNEUMONIA, INTRACTABLE EPILEPSI.md ===


458
BAYI DENGAN NEONATUS ATERM + BERAT BADAN LAHIR RENDAH .md ===


3727
BAYI PREMATUR BBLR SMK (SESUAI MASA KEHAMILAN) LAHIR SC 3334 MINGGU.md ===


2321
BBLR, OEIS COMPLEX (OMPHALOCELE + EXTROFIA CLOACA + IMPERFRATA ANUS.md ===


1365
BERAT BADAN LAHIR SANGAT RENDAH (BBLSR), SESUAI MASA KEHAMILAN (SMK).md ===


1915
BERAT BADAN LAHIR SANGAT RENDAH (BBSLR), NEONATUS VERY PETERM (NVP).md ===


481
BRONKOPNEUMONIA + GIZI KURANG + ANEMIA.md ===


1995
BRONKOPNEUMONIA.md ===


5586
BURUK MARASMUS, STUNTING, PNEUMONIA, ASD SEKUNDER SEDANG,.md ===


5469
CERBRAL PALSY + HIDROSEFALUS DENGAN VP SHUNT + GIZI BURUK + BRONKITIS.md ===


3251
CEREBRAL PALSY, GLOBAL DEVE

In [29]:
counter

[]

## Extract Fields via LLM

In [ ]:
from google import genai
from pydantic import BaseModel, Field
from typing import Optional, List

In [ ]:
class MedicalExtraction(BaseModel):
    anthropometric_assessment: str = Field(..., description="Extract comprehensive anthropometric data including Gender, Age, Weight (BB), Height (TB/PB), Head Circumference (Optional), Mid-Upper Arm Circumference (LiLa) (Optional), Z-scores (BB/U, PB/U, BB/PB) (Optional), and any interpretation of these values.")
    medical_diagnosis: str = Field(..., description="Extract the primary and secondary medical diagnoses, including differential diagnoses if present.")